In [1]:
import matplotlib.pyplot as plt
import os
import numpy as np
from monai.transforms import LoadImage, EnsureChannelFirst, Orientation, Compose
from monai.bundle import ConfigParser, download
from monai.data.meta_tensor import MetaTensor
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
import seaborn as sns
from ipywidgets import interact
from nibabel import Nifti1Image

from monai.transforms import Resize



In [14]:
%matplotlib inline

def show_scrollable_image(image, cmap="nipy_spectral"):
    """
    Displays a 3D array as a scrollable series of slices.
    
    Parameters:
    - image (numpy.ndarray): A 3D numpy array (e.g., (H, W, D)) representing the image.
    """
    
    # Define a function to display each slice
    def display_slice(slice_index):
        plt.figure(figsize=(8, 8))
        plt.imshow(image[:, :, slice_index], cmap=cmap)
        plt.colorbar(label='HU')
        plt.axis("off")
        plt.title(f"Slice {slice_index + 1} / {image.shape[2]}")
        plt.show()
    
    # Create an interactive slider to scroll through slices
    interact(display_slice, slice_index=(0, image.shape[2] - 1))

In [3]:
import os

# Define the path to the main directory
testing_subset_path = '../data/Inputs/takotsubo_cases/testing_subset/'

min_slices = float('inf')

# Loop through all folders in the directory
for folder_name in sorted(os.listdir(testing_subset_path)):
    folder_path = os.path.join(testing_subset_path, folder_name)
    
    # Check if the path is a directory
    if os.path.isdir(folder_path):
        print(folder_name)
    
        dicom_files_folder = os.path.join(folder_path, "dicom", "EXP00000")
        print(dicom_files_folder)
        num_slices = len(os.listdir(dicom_files_folder))
        min_slices = min(min_slices, num_slices)
        print(num_slices)
        
        # for dicom_file in sorted(os.listdir(dicom_files_folder)):
        #     print(f"    {dicom_file}")
print(f"Min_slices: {min_slices}")  

JNMP 02291631
../data/Inputs/takotsubo_cases/testing_subset/JNMP 02291631/dicom/EXP00000
151
JNMP 2nd scan
../data/Inputs/takotsubo_cases/testing_subset/JNMP 2nd scan/dicom/EXP00000
96
JOS 3812401
../data/Inputs/takotsubo_cases/testing_subset/JOS 3812401/dicom/EXP00000
142
LET 04456356
../data/Inputs/takotsubo_cases/testing_subset/LET 04456356/dicom/EXP00000
982
LMA 03884426
../data/Inputs/takotsubo_cases/testing_subset/LMA 03884426/dicom/EXP00000
334
LMC 03629185
../data/Inputs/takotsubo_cases/testing_subset/LMC 03629185/dicom/EXP00000
210
LMD 22194237
../data/Inputs/takotsubo_cases/testing_subset/LMD 22194237/dicom/EXP00000
344
LW 12109724
../data/Inputs/takotsubo_cases/testing_subset/LW 12109724/dicom/EXP00000
235
MC 03887643
../data/Inputs/takotsubo_cases/testing_subset/MC 03887643/dicom/EXP00000
159
MDW 2286750
../data/Inputs/takotsubo_cases/testing_subset/MDW 2286750/dicom/EXP00000
111
MG 22519714
../data/Inputs/takotsubo_cases/testing_subset/MG 22519714/dicom/EXP00000
92
MGR 025

In [4]:
# dicom_file_path = 'data/normal_cases/AG 519880_37F/DICOM/EXP00000/'
# dicom_file_path = 'data/normal_cases/ASJ 05633540_68F/DICOM/EXP00000/'
# dicom_file_path = '../data/takotsubo_cases/testing_subset/JNMP 02291631/DICOM/EXP00000/'
# dicom_file_path = '../data/takotsubo_cases/testing_subset/JOS 3812401/DICOM/EXP00000/'

input_folder = "../data/Inputs/takotsubo_cases/testing_subset/LET 04456356"
dicom_file_path = os.path.join(input_folder, "DICOM", "EXP00000")

output_folder = "../data/Outputs/takotsubo_cases/testing_subset/LET 04456356"

image_loader = LoadImage(image_only=True)
original_input_image = image_loader(dicom_file_path)

original_input_image.meta

{'00200037': {'vr': 'DS', 'Value': [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]},
 '00200032': {'vr': 'DS', 'Value': [-195.7, -175.7, 50.0]},
 '00280030': {'vr': 'DS', 'Value': [0.806641, 0.806641, 0.625]},
 'spacing': array([0.806641, 0.806641, 0.625   ]),
 'lastImagePositionPatient': array([-195.7  , -175.7  , -563.125]),
 spatial_shape: (512, 512, 982),
 space: RAS,
 original_affine: array([[ -0.806641,   0.      ,   0.      , 195.7     ],
        [  0.      ,  -0.806641,   0.      , 175.7     ],
        [  0.      ,   0.      ,  -0.625   ,  50.      ],
        [  0.      ,   0.      ,   0.      ,   1.      ]]),
 affine: tensor([[ -0.8066,   0.0000,   0.0000, 195.7000],
         [  0.0000,  -0.8066,   0.0000, 175.7000],
         [  0.0000,   0.0000,  -0.6250,  50.0000],
         [  0.0000,   0.0000,   0.0000,   1.0000]], dtype=torch.float64),
 original_channel_dim: nan,
 'filename_or_obj': '../data/Inputs/takotsubo_cases/testing_subset/LET 04456356/DICOM/EXP00000'}

In [5]:
show_scrollable_image(original_input_image, cmap="grey")

interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [6]:
import torch
print(torch.backends.mps.is_available())  # Should return True
print(torch.backends.mps.is_built())  # Should return True


True
True


In [7]:
import nibabel as nib
from totalsegmentator.python_api import totalsegmentator


# option 1: provide input and output as file paths
# output_directory = "../data/segment_eval_output_takotsubo_case2_bodyseg"
# output_image = totalsegmentator(dicom_file_path, output_folder, license_number="aca_BWYHC6UQQFDU8A", task="heartchambers_highres", body_seg=True)
output_image = totalsegmentator(dicom_file_path, output_folder, license_number="aca_BWYHC6UQQFDU8A", roi_subset=["heart"], device="mps")


# option 2: provide input and output as nifti image objects
# input_img = nib.load(dicom_file_path)
# output_img = totalsegmentator(input_img)
# print(output_img)
# nib.save(output_img, data_directory)


If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Converting dicom to nifti...
  found image with shape (512, 512, 982)
Resampling...
  Resampled in 8.12s
Predicting...


100%|██████████| 12/12 [00:02<00:00,  5.25it/s]


  Predicted in 8.49s
Resampling...
Converting dicom to nifti...
  found image with shape (512, 512, 982)
  cropping from (512, 512, 982) to (213, 176, 288)
Resampling...
  Resampled in 0.62s
Predicting part 1 of 1 ...


100%|██████████| 1/1 [00:02<00:00,  2.71s/it]


  Predicted in 10.72s
Resampling...
Saving segmentations...
  Saved in 11.36s


In [8]:
def nifti1image_to_metatensor(nifti_img: Nifti1Image) -> MetaTensor:
    """
    Converts a Nifti1Image object to a MetaTensor with metadata.
    Args:
        nifti_img (Nifti1Image): A loaded NIfTI image.
    Returns:
        MetaTensor: A MONAI MetaTensor containing the image data and metadata.
    """
    # Convert voxel data to PyTorch tensor
    image_tensor = torch.tensor(nifti_img.get_fdata(), dtype=torch.float32)

    # Extract metadata
    metadata = {
        "affine": torch.tensor(nifti_img.affine, dtype=torch.float32),  # Affine transformation matrix
        "spacing": nifti_img.header.get_zooms(),  # Voxel spacing (x, y, z)
        "original_shape": image_tensor.shape  # Store original shape for reference
    }

    # Create a MetaTensor with metadata
    meta_tensor = MetaTensor(image_tensor, meta=metadata)

    return meta_tensor


In [23]:
show_scrollable_image(nifti1image_to_metatensor(output_image))

interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [9]:
# nib.save(output_image, f'{output_folder}/output_image.nii.gz')

In [9]:
full_image = image_loader(f"{output_folder}/output_image.nii.gz")
show_scrollable_image(full_image)

interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [10]:
# Load the image
left_ventricle_img = image_loader(f"{output_folder}/heart_ventricle_left.nii.gz")

# Check the image type and shape
print(type(left_ventricle_img))
print(left_ventricle_img.shape)

show_scrollable_image(left_ventricle_img)

<class 'monai.data.meta_tensor.MetaTensor'>
torch.Size([512, 512, 982])


interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [ ]:
def resize_to_target_slices(image: MetaTensor, heart_mask: MetaTensor, target_width=512, target_height=512, target_slices=128):
    """
    Extracts the heart region from the input scan and resizes it to a target number of slices, 
    preserving metadata while correctly computing voxel spacing.

    Args:
        image (MetaTensor): The input 3D CT/MRI scan.
        heart_mask (MetaTensor): Binary mask indicating heart region.
        target_slices (int, optional): Number of slices to resize to. Default is 128.

    Returns:
        MetaTensor: Resized image with updated metadata.
    """
    if isinstance(heart_mask, MetaTensor):
        heart_mask = heart_mask.as_tensor().cpu().numpy()
    
    z_indices = np.any(heart_mask > 0, axis=(0, 1))
    heart_slices = np.where(z_indices)[0]
    # print(heart_slices)
    if len(heart_slices) == 0:
        raise ValueError("No heart region found in the mask.")
    
    start_slice = heart_slices[0]
    end_slice = heart_slices[-1] + 1
    
    sliced_image = image[:, :, start_slice:end_slice]
    if not isinstance(sliced_image, MetaTensor):
        sliced_image = MetaTensor(sliced_image, meta=image.meta)
    
    height, width, original_depth = image.shape
    # sliced_depth = sliced_image.shape[-1]

    sliced_image = sliced_image.unsqueeze(0)  # Shape becomes (1, H, W, Z)
    
    resizer = Resize(spatial_size=(target_width,target_height,target_slices), mode="trilinear", align_corners=True)
    resized_image = resizer(sliced_image)
    
    resized_image = resized_image.squeeze(0)

    # Update metadata
    new_meta = image.meta.copy()  
    
    original_spacing = image.meta.get("spacing", (1.0, 1.0, 1.0))
    new_spacing = (
        original_spacing[0],
        original_spacing[1],
        original_spacing[2] * (original_depth / target_slices)
    )
    new_meta["spacing"] = new_spacing

    # Keep affine transformation matrix but adjust slice resolution
    if "affine" in new_meta:
        new_meta["affine"][-1, -1] *= (original_depth / target_slices)  # Adjust Z scaling

    # Return resized MetaTensor with updated metadata
    return MetaTensor(resized_image, meta=new_meta)

    

full_image = nifti1image_to_metatensor(output_image)
resized_image = resize_to_target_slices(full_image, full_image)



print("Resized Shape:", resized_image.shape)  # (H, W, 128)
print("Updated Voxel Spacing:", resized_image.meta["spacing"])
print("Updated Affine Matrix:\n", resized_image.meta["affine"])


Resized Shape: torch.Size([512, 512, 128])
Updated Voxel Spacing: (0.806641, 0.806641, 4.794921875)
Updated Affine Matrix:
 tensor([[  -0.8066,    0.0000,    0.0000,  195.7000],
        [  -0.0000,    0.8066,    0.0000, -236.4935],
        [   0.0000,   -0.0000,    0.6250, -563.1250],
        [   0.0000,    0.0000,    0.0000,    7.6719]], dtype=torch.float64)


In [16]:
print(resized_image.shape)


torch.Size([512, 512, 128])


In [25]:
show_scrollable_image(resized_image, cmap="grey")

interactive(children=(IntSlider(value=63, description='slice_index', max=127), Output()), _dom_classes=('widge…

In [16]:
def align_segmentation_to_image(segmentation):
    """
    Aligns the TotalSegmentator output (segmentation) with the original image
    by flipping the z-axis of the segmentation mask.
    
    Parameters:
        image (torch.Tensor or np.ndarray): Original image (H x W x D).
        segmentation (torch.Tensor or np.ndarray): TotalSegmentator output (H x W x D).
        
    Returns:
        torch.Tensor: Segmentation aligned with the image.
    """
   
    # Flip along the z-axis (last dimension)
    aligned_segmentation = torch.flip(segmentation, dims=[-1])
    
    return aligned_segmentation

# Example usage
aligned_mask = align_segmentation_to_image(full_image)



In [17]:
show_scrollable_image(aligned_mask)

interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [18]:
original_image_resized = resize_to_target_slices(original_input_image, aligned_mask)

[138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155
 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173
 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191
 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209
 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227
 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245
 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263
 264 265 266 267 268 269 270 271 272 273 274 275 276 277 278 279 280 281
 282 283 284 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299
 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317
 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335
 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353
 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371
 372 373 374 375 376 377 378 379 380 381 382 383 38

In [20]:
show_scrollable_image(original_image_resized, cmap="grey")

interactive(children=(IntSlider(value=63, description='slice_index', max=127), Output()), _dom_classes=('widge…